# Replication: Fong, Hazlett, and Imai (2018)

**Covariate Balancing Propensity Score for a Continuous Treatment: Application to the Efficacy of Political Advertisements**  
*The Annals of Applied Statistics*, 12(1), 156–177.

This notebook replicates the main simulation study and empirical application from the paper:

- **Section 4** — Simulation study comparing four estimators under four DGPs (Figures 1–2)
- **Section 5** — Empirical application: effect of political advertising on vote share (Table 1, Figure 3)

In [1]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

import cbps
from cbps.datasets import load_political_ads

## Part 1: Simulation Study (Section 4, Figures 1–2)

Four DGPs cross correct/incorrect specification of the treatment model and the outcome model. Ten covariates are drawn from $\mathcal{N}(0, \Sigma)$ with $\Sigma_{jk} = 0.2$ for $j \neq k$. The true ATE is 1.0.

| DGP | Treatment model | Outcome model |
|-----|----------------|---------------|
| 1   | Correct        | Correct       |
| 2   | Misspecified   | Correct       |
| 3   | Correct        | Misspecified  |
| 4   | Misspecified   | Misspecified  |

### Data Generating Process

In [2]:
COV_COLS = [f'X{j}' for j in range(1, 11)]
SIM_FORMULA = 'T ~ ' + ' + '.join(COV_COLS)

DGP_LABELS = {
    1: 'DGP1: Both correctly specified',
    2: 'DGP2: Treatment misspecified',
    3: 'DGP3: Outcome misspecified',
    4: 'DGP4: Both misspecified',
}


def generate_dgp(dgp, n=200, rng=None):
    """One Monte Carlo draw following Section 4.1."""
    rng = rng or np.random.default_rng()
    K = 10
    Sigma = np.full((K, K), 0.2)
    np.fill_diagonal(Sigma, 1.0)
    X = rng.multivariate_normal(np.zeros(K), Sigma, size=n)

    if dgp in (1, 3):
        T = X[:,0] + X[:,1] + 0.2*X[:,2] + 0.2*X[:,3] + 0.2*X[:,4] + rng.normal(0, 2.0, n)
    else:
        T = (X[:,1]+0.5)**2 + 0.4*X[:,2] + 0.4*X[:,3] + 0.4*X[:,4] + rng.normal(0, 1.5, n)

    if dgp in (1, 2):
        Y = X[:,1] + 0.1*X[:,3] + 0.1*X[:,4] + 0.1*X[:,5] + T + rng.normal(0, 5.0, n)
    else:
        Y = 2*(X[:,1]+0.5)**2 + T + 0.5*X[:,3] + 0.5*X[:,4] + 0.5*X[:,5] + rng.normal(0, 5.0, n)

    df = pd.DataFrame(X, columns=COV_COLS)
    df['T'], df['Y'] = T, Y
    return df

### Helper Functions

In [3]:
def ate_wls(df, weights=None):
    """WLS regression of Y on T; returns the slope (ATE estimate)."""
    Xr = sm.add_constant(df['T'].values)
    m = sm.WLS(df['Y'].values, Xr, weights=weights) if weights is not None else sm.OLS(df['Y'].values, Xr)
    return m.fit().params[1]


def f_balance(T, X, weights=None):
    """F-statistic from (weighted) regression of T on X."""
    Xc = sm.add_constant(X)
    n = len(T)
    if weights is not None:
        w = weights * n / weights.sum()
        return sm.WLS(T, Xc, weights=w).fit().fvalue
    return sm.OLS(T, Xc).fit().fvalue


def wcorr(T, X, weights=None):
    """Weighted Pearson correlations between T and each column of X."""
    K = X.shape[1]
    out = np.zeros(K)
    if weights is None:
        for j in range(K):
            out[j] = np.corrcoef(T, X[:, j])[0, 1]
    else:
        w = weights / weights.sum()
        mT = w @ T
        for j in range(K):
            mX = w @ X[:, j]
            cov = np.sum(w * (T - mT) * (X[:, j] - mX))
            vT = np.sum(w * (T - mT)**2)
            vX = np.sum(w * (X[:, j] - mX)**2)
            out[j] = cov / np.sqrt(vT * vX) if vT > 0 and vX > 0 else 0.0
    return out

### Run Simulation

In [4]:
n_sim = 20  # Paper uses 500; reduced for computational tractability
seed = 42
methods = ['Unadjusted', 'MLE', 'CBGPS', 'npCBGPS']

R = {d: {m: {'f': [], 'ate': []} for m in methods} for d in range(1, 5)}
rng = np.random.default_rng(seed)

for dgp in range(1, 5):
    print(f'  {DGP_LABELS[dgp]}')
    for _ in range(n_sim):
        df_sim = generate_dgp(dgp, rng=rng)
        T, X = df_sim['T'].values, df_sim[COV_COLS].values

        # Unadjusted
        R[dgp]['Unadjusted']['ate'].append(ate_wls(df_sim))
        R[dgp]['Unadjusted']['f'].append(f_balance(T, X))

        # MLE (standard GPS via OLS, no balance optimization)
        try:
            Xc_sim = sm.add_constant(X)
            ols_sim = sm.OLS(T, Xc_sim).fit()
            gps_sim = norm.pdf(T, loc=ols_sim.fittedvalues,
                               scale=np.sqrt(np.mean(ols_sim.resid**2)))
            marg_sim = norm.pdf(T, loc=T.mean(), scale=T.std(ddof=1))
            w_sim = marg_sim / np.clip(gps_sim, 1e-6, None)
            w_sim = w_sim / w_sim.sum()
            R[dgp]['MLE']['ate'].append(ate_wls(df_sim, w_sim))
            R[dgp]['MLE']['f'].append(f_balance(T, X, w_sim))
        except Exception:
            R[dgp]['MLE']['ate'].append(np.nan)
            R[dgp]['MLE']['f'].append(np.nan)

        # CBGPS (parametric, just-identified per paper eq. 2)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                fit = cbps.CBPS(formula=SIM_FORMULA, data=df_sim, att=0,
                                method='exact', verbose=0)
            R[dgp]['CBGPS']['ate'].append(ate_wls(df_sim, fit.weights))
            R[dgp]['CBGPS']['f'].append(f_balance(T, X, fit.weights))
        except Exception:
            R[dgp]['CBGPS']['ate'].append(np.nan)
            R[dgp]['CBGPS']['f'].append(np.nan)

        # npCBGPS (nonparametric)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                fit = cbps.npCBPS(formula=SIM_FORMULA, data=df_sim,
                                  corprior=0.1, print_level=0)
            R[dgp]['npCBGPS']['ate'].append(ate_wls(df_sim, fit.weights))
            R[dgp]['npCBGPS']['f'].append(f_balance(T, X, fit.weights))
        except Exception:
            R[dgp]['npCBGPS']['ate'].append(np.nan)
            R[dgp]['npCBGPS']['f'].append(np.nan)

print('Done.')

  DGP1: Both correctly specified
  DGP2: Treatment misspecified
  DGP3: Outcome misspecified
  DGP4: Both misspecified
Done.


### Simulation Results

In [5]:
def print_sim_table(title, key, func):
    print(f'\n{title}')
    print(f"{'DGP':<30s}" + ''.join(f'{m:>12s}' for m in methods))
    for d in range(1, 5):
        row = DGP_LABELS[d].split(':')[0]
        vals = []
        for m in methods:
            v = [x for x in R[d][m][key] if np.isfinite(x)]
            vals.append(func(v) if v else np.nan)
        print(f'{row:<30s}' + ''.join(f'{v:>12.4f}' for v in vals))


print_sim_table('Median F-statistic (cf. Figure 1)', 'f', np.median)
print_sim_table('ATE bias = E[ATE_hat] - 1.0 (cf. Figure 2)', 'ate',
                lambda v: np.mean(v) - 1.0)
print_sim_table('ATE RMSE (cf. Figure 2)', 'ate',
                lambda v: np.sqrt(np.mean((np.array(v) - 1.0)**2)))


Median F-statistic (cf. Figure 1)
DGP                             Unadjusted         MLE       CBGPS     npCBGPS
DGP1                               16.4999      2.3313      0.7953      0.0000
DGP2                                9.5633     27.0292      0.0946      0.0000
DGP3                               15.8140      2.2735      1.2125      0.0000
DGP4                               10.7441     11.9685      0.1587      0.0000

ATE bias = E[ATE_hat] - 1.0 (cf. Figure 2)
DGP                             Unadjusted         MLE       CBGPS     npCBGPS
DGP1                                0.2114     -0.0024      0.0862      0.0270
DGP2                                0.2834     -0.4430     -0.0051      0.0201
DGP3                                0.4830      0.0922      0.1434     -0.0774
DGP4                                1.2169      1.2816      1.0419      1.0401

ATE RMSE (cf. Figure 2)
DGP                             Unadjusted         MLE       CBGPS     npCBGPS
DGP1                       

## Part 2: Empirical Application (Section 5, Table 1, Figure 3)

Data from Urban and Niebler (2014). Treatment: total political advertisements (TotAds), Box–Cox transformed with $\lambda = -0.16$. Outcome: Democratic vote share.

### Data Preparation

In [6]:
COV_DISPLAY = [
    ('logPop',            'log(Population)'),
    ('density',           'Population density'),
    ('logInc',            'log(Income+1)'),
    ('PercentHispanic',   '% Hispanic'),
    ('PercentBlack',      '% Black'),
    ('PercentOver65',     '% Over 65'),
    ('per_collegegrads',  '% College graduates'),
    ('CanCommute',        'Commute indicator'),
    ('logPop_sq',         'log(Population)^2'),
    ('density_sq',        'Population density^2'),
    ('logInc_sq',         'log(Income+1)^2'),
    ('PercentHispanic_sq', '% Hispanic^2'),
    ('PercentBlack_sq',   '% Black^2'),
    ('PercentOver65_sq',  '% Over 65^2'),
    ('per_collegegrads_sq', '% College graduates^2'),
]

df_raw, meta = load_political_ads()
print(f'Dataset: Urban & Niebler (2014), n={len(df_raw)}')
print(f'Treatment: TotAds, Box-Cox lambda={meta["boxcox_lambda"]}')

work = df_raw.copy()
lam = meta['boxcox_lambda']
work['T_bc'] = ((work['TotAds'].values + 1).clip(min=1e-10)**lam - 1.0) / lam
work['logPop'] = np.log(work['TotalPop'].values.clip(min=1))
work['logInc'] = np.log(work['Inc'].values.clip(min=0) + 1)

# Add squared terms for all non-binary covariates (paper p. 171)
work['logPop_sq'] = work['logPop'] ** 2
work['density_sq'] = work['density'] ** 2
work['logInc_sq'] = work['logInc'] ** 2
work['PercentHispanic_sq'] = work['PercentHispanic'] ** 2
work['PercentBlack_sq'] = work['PercentBlack'] ** 2
work['PercentOver65_sq'] = work['PercentOver65'] ** 2
work['per_collegegrads_sq'] = work['per_collegegrads'] ** 2

cov_cols = [c for c, _ in COV_DISPLAY]
work = work.dropna(subset=['T_bc'] + cov_cols).reset_index(drop=True)
print(f'Analysis sample: n={len(work)}')

formula = 'T_bc ~ ' + ' + '.join(cov_cols)
T = work['T_bc'].values
X = work[cov_cols].values

Dataset: Urban & Niebler (2014), n=16265
Treatment: TotAds, Box-Cox lambda=-0.16
Analysis sample: n=16265


### Fit Estimators

In [7]:
corrs, fstats = {}, {}

# Unweighted
corrs['Unweighted'] = wcorr(T, X)
fstats['Unweighted'] = f_balance(T, X)

# MLE (standard GPS via OLS)
print('Fitting MLE (standard GPS)...')
Xc_mle = sm.add_constant(X)
ols_fit = sm.OLS(T, Xc_mle).fit()
T_hat = ols_fit.fittedvalues
sigma_mle = np.sqrt(np.mean(ols_fit.resid**2))
gps_mle = norm.pdf(T, loc=T_hat, scale=sigma_mle)
marginal_mle = norm.pdf(T, loc=T.mean(), scale=T.std(ddof=1))
w_mle = marginal_mle / np.clip(gps_mle, 1e-6, None)
w_mle = w_mle / w_mle.sum()
corrs['MLE'] = wcorr(T, X, w_mle)
fstats['MLE'] = f_balance(T, X, w_mle)

# CBGPS (parametric)
print('Fitting CBGPS (parametric)...')
fit_cb = cbps.CBPS(formula=formula, data=work, att=0,
                   method='exact', verbose=0)
corrs['CBGPS'] = wcorr(T, X, fit_cb.weights)
fstats['CBGPS'] = f_balance(T, X, fit_cb.weights)

# npCBGPS (nonparametric)
print('Fitting npCBGPS (nonparametric)...')
fit_np = cbps.npCBPS(formula=formula, data=work,
                     corprior=0.1, print_level=0)
corrs['npCBGPS'] = wcorr(T, X, fit_np.weights)
fstats['npCBGPS'] = f_balance(T, X, fit_np.weights)

print('Done.')

Fitting MLE (standard GPS)...
Fitting CBGPS (parametric)...


cbps/__init__.py:999: UserWarning: Parameter conflict: method='exact' (just-identified GMM) is incompatible with two_step=True (requires over-identification). 

Explanation:
  - method='exact': Uses only balance conditions (#params = #moments)
  - two_step=True: Requires both score and balance conditions (overidentified)

Action: The two_step parameter will be ignored. Use method='over' if you want two-step estimation.
  warnings.warn(


Fitting npCBGPS (nonparametric)...
Done.


### Table 1: Signed Pearson Correlations (p. 171)

In [8]:
est_names = ['Unweighted', 'MLE', 'CBGPS', 'npCBGPS']

print(f"{'Covariate':<25s}" + ''.join(f'{e:>12s}' for e in est_names))
for j, (col, label) in enumerate(COV_DISPLAY):
    row = f'{label:<25s}'
    for e in est_names:
        row += f'{corrs[e][j]:>12.3f}'
    print(row)

Covariate                  Unweighted         MLE       CBGPS     npCBGPS
log(Population)                0.0592      0.0023      0.0001      0.0008
Population density             0.0883      0.0280      0.0001      0.0054
log(Income+1)                  0.0123      0.0176      0.0000      0.0027
% Hispanic                     0.0426      0.0044      0.0000      0.0035
% Black                        0.0763      0.0032      0.0000      0.0024
% Over 65                      0.0058      0.0019      0.0000      0.0003
% College graduates            0.0324      0.0036      0.0000      0.0029
Commute indicator              0.0543      0.0034      0.0000      0.0008


### Figure 3: F-statistic (p. 172)

In [9]:
print('F-statistic from regression of T on X')
print(f"  Unweighted:  {fstats['Unweighted']:.4f}  (paper: ~29.3)")
print(f"  MLE:         {fstats['MLE']:.4f}")
print(f"  CBGPS:       {fstats['CBGPS']:.6f}  (paper: ~9.33e-5)")
print(f"  npCBGPS:     {fstats['npCBGPS']:.6f}")

F-statistic from regression of T on X
  Unweighted:  35.6482  (paper: ~29.3)
  MLE:         2.2068
  CBGPS:       0.000069  (paper: ~9.33e-5)
  npCBGPS:     0.105463


### Convergence Diagnostics

In [10]:
# CBGPS diagnostics — now included in summary() output
print('=== CBGPS Summary ===')
print(f'Converged: {fit_cb.converged}, J={fit_cb.J:.6f}')
print(f'Weights: [{fit_cb.weights.min():.4f}, {fit_cb.weights.max():.4f}]')
print()

# npCBPS now returns NPCBPSSummary object with rich diagnostics
print('=== npCBGPS Summary ===')
summary_np = fit_np.summary()
print(f'Summary type: {type(summary_np).__name__}')
print(summary_np)

print()
print('Note: Simulation uses 20 replications (paper uses 500).')
print('Exact numerical agreement is not expected due to differences in')
print('random draws and optimizer implementations between R and Python.')

CBGPS converged: True, J=0.000000
CBGPS weights: [0.0000, 0.0002]
npCBGPS weights: [0.3840, 7.1547]

Note: Simulation uses 20 replications (paper uses 500).
Exact numerical agreement is not expected due to differences in
random draws and optimizer implementations between R and Python.
